# ReferSplat: Referring Segmentation in 3D Gaussian Splatting
**ICML 2025 Oral**

이 노트북은 ReferSplat을 Google Colab 환경에서 실행하기 위한 전체 파이프라인입니다.

**파이프라인:**
1. 환경 설정 및 의존성 설치
2. 데이터셋 준비 (LERF-OVS)
3. 사전학습 3DGS 체크포인트 준비
4. ReferSplat 학습 (Stage 1)
5. 렌더링 (Inference)
6. 평가 (mIoU)

## 0. GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. 환경 설정

### 1-1. 경로 설정
**여기서 데이터 경로를 수정하세요.**

In [ ]:
import os

# ============================================================
# [수정 필요] 데이터 및 출력 경로 설정
# ============================================================
DATA_ROOT = "/content/data/lerf-ovs/Ref-lerf"  # LERF-OVS 데이터셋 루트 경로
OUTPUT_ROOT = "/content/output"                # 학습 결과 저장 경로
CHECKPOINT_ROOT = "/content/checkpoints"       # 사전학습 3DGS 체크포인트 경로
REPO_DIR = "/content/ReferSplat"               # 레포 클론 경로
GITHUB_REPO = "BAEJUNHAK/ReferSplat"          # GitHub 레포 (username/repo)

# 사용할 장면 선택 (figurines, ramen, teatime, waldo_kitchen)
SCENE = "ramen"

# 파생 경로 (자동 설정)
SCENE_DATA_PATH = os.path.join(DATA_ROOT, SCENE)
SCENE_OUTPUT_PATH = os.path.join(OUTPUT_ROOT, SCENE)
PRETRAINED_CKPT = os.path.join(CHECKPOINT_ROOT, SCENE, "chkpnt30000.pth")  # 3DGS RGB 체크포인트

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)

print(f"Scene: {SCENE}")
print(f"Data path: {SCENE_DATA_PATH}")
print(f"Output path: {SCENE_OUTPUT_PATH}")
print(f"Pretrained checkpoint: {PRETRAINED_CKPT}")
print(f"GitHub repo: {GITHUB_REPO}")

### 1-2. 레포지토리 클론 및 CUDA 서브모듈 빌드

In [ ]:
# 레포지토리 클론 (자신의 GitHub 레포에서)
%cd /content
if not os.path.exists(REPO_DIR):
    !git clone --recursive https://github.com/{GITHUB_REPO}.git
else:
    # 이미 클론된 경우 최신 변경사항 pull
    print("Repository already exists, pulling latest changes...")
    !cd {REPO_DIR} && git pull && git submodule update --init --recursive
%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# 필수 패키지 설치
!pip install plyfile==0.8.1 jaxtyping open-clip-torch tensorboard -q

In [ ]:
# transformers 설치 (BERT 사용)
# 원본은 transformers==3.2.0이지만 Colab Python 3.10+와 호환 안됨
# BertTokenizer/BertModel API는 호환되므로 최신 버전 사용
!pip install transformers -q

In [ ]:
# CUDA 서브모듈 빌드: langsplat-rasterization
print("=" * 50)
print("Building langsplat-rasterization (CUDA)...")
print("=" * 50)
%cd {REPO_DIR}/submodules/langsplat-rasterization
!pip install --no-build-isolation .
%cd {REPO_DIR}

In [ ]:
# CUDA 서브모듈 빌드: simple-knn
print("=" * 50)
print("Building simple-knn (CUDA)...")
print("=" * 50)
%cd {REPO_DIR}/submodules/simple-knn
!pip install --no-build-isolation .
%cd {REPO_DIR}

In [ ]:
# segment-anything-langsplat 빌드
print("=" * 50)
print("Building segment-anything-langsplat...")
print("=" * 50)
%cd {REPO_DIR}/submodules/segment-anything-langsplat
!pip install --no-build-isolation .
%cd {REPO_DIR}

In [ ]:
# 빌드 검증
import sys
sys.path.insert(0, REPO_DIR)

try:
    import diff_gaussian_rasterization
    print("[OK] diff_gaussian_rasterization")
except ImportError as e:
    print(f"[FAIL] diff_gaussian_rasterization: {e}")

try:
    from simple_knn._C import distCUDA2
    print("[OK] simple_knn")
except ImportError as e:
    print(f"[FAIL] simple_knn: {e}")

try:
    from transformers import BertTokenizer, BertModel
    print("[OK] transformers (BERT)")
except ImportError as e:
    print(f"[FAIL] transformers: {e}")

## 2. 데이터셋 다운로드

LERF-OVS 데이터셋 (4개 장면: figurines, ramen, teatime, waldo_kitchen)

**다운로드 옵션:**
- HuggingFace: https://huggingface.co/datasets/FudanCVL/Ref-Lerf
- Baidu: https://pan.baidu.com/s/1D9yDNfUrK-d8eGO33Avkpg?pwd=ugs3

**데이터 구조 (각 장면별):**
```
<scene>/
├── images/              # RGB 이미지
├── sparse/0/            # COLMAP (cameras.bin, images.bin, points3D.bin)
├── json/
│   ├── train_json/      # frame_XXXXX.json (학습 어노테이션)
│   └── test_json/       # frame_XXXXX.json (테스트 어노테이션)
└── gt_mask/             # GT 세그멘테이션 마스크 (PNG)
```

In [ ]:
# ============================================================
# 방법 1: HuggingFace에서 다운로드
# ============================================================
# !pip install huggingface_hub -q
# from huggingface_hub import snapshot_download
# snapshot_download(
#     repo_id="FudanCVL/Ref-Lerf",
#     repo_type="dataset",
#     local_dir=DATA_ROOT,
# )

# ============================================================
# 방법 2: Google Drive에서 직접 업로드 / 마운트
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = "/content/drive/MyDrive/path/to/lerf-ovs"  # 경로 수정

# ============================================================
# 방법 3: 이미 데이터가 있는 경우 경로만 확인
# ============================================================
print("데이터 경로 확인:")
if os.path.exists(SCENE_DATA_PATH):
    print(f"  [OK] {SCENE_DATA_PATH}")
    !ls {SCENE_DATA_PATH}
else:
    print(f"  [NOT FOUND] {SCENE_DATA_PATH}")
    print("  위 방법 중 하나를 사용해서 데이터를 다운로드하세요.")

## 3. 체크포인트 다운로드

사전학습된 3DGS RGB 체크포인트가 필요합니다.

**다운로드:**
- Google Drive: https://drive.google.com/drive/folders/1z9O2FWwUlE29lSgLDj9Af7sv5ZQv6sc_
- HuggingFace: https://huggingface.co/FudanCVL/RefSplat

In [ ]:
# ============================================================
# 방법 1: HuggingFace에서 체크포인트 다운로드
# ============================================================
# !pip install huggingface_hub -q
# from huggingface_hub import snapshot_download
# snapshot_download(
#     repo_id="FudanCVL/RefSplat",
#     local_dir=CHECKPOINT_ROOT,
# )

# ============================================================
# 방법 2: gdown으로 Google Drive에서 다운로드
# ============================================================
# !pip install gdown -q
# !gdown --folder "https://drive.google.com/drive/folders/1z9O2FWwUlE29lSgLDj9Af7sv5ZQv6sc_" -O {CHECKPOINT_ROOT}

print("체크포인트 경로 확인:")
if os.path.exists(PRETRAINED_CKPT):
    print(f"  [OK] {PRETRAINED_CKPT}")
else:
    print(f"  [NOT FOUND] {PRETRAINED_CKPT}")
    print("  위 방법 중 하나를 사용해서 체크포인트를 다운로드하세요.")
    print(f"  체크포인트 디렉토리 내용:")
    !ls -R {CHECKPOINT_ROOT} 2>/dev/null || echo "  디렉토리 없음"

## 4. 데이터 구조 검증

In [ ]:
import json
from pathlib import Path

def verify_scene(scene_path):
    """데이터셋 구조 검증"""
    checks = {
        "images/": os.path.isdir(os.path.join(scene_path, "images")),
        "sparse/0/": os.path.isdir(os.path.join(scene_path, "sparse", "0")),
        "cameras.bin": os.path.isfile(os.path.join(scene_path, "sparse", "0", "cameras.bin")),
        "images.bin": os.path.isfile(os.path.join(scene_path, "sparse", "0", "images.bin")),
        "points3D.bin": os.path.isfile(os.path.join(scene_path, "sparse", "0", "points3D.bin")),
        "gt_mask/": os.path.isdir(os.path.join(scene_path, "gt_mask")),
    }
    # json 폴더는 json/ 아래 또는 직접
    has_json = os.path.isdir(os.path.join(scene_path, "json")) or \
               os.path.isdir(os.path.join(scene_path, "train_json"))
    checks["json annotations"] = has_json

    print(f"\n--- Scene: {os.path.basename(scene_path)} ---")
    all_ok = True
    for name, ok in checks.items():
        status = "OK" if ok else "MISSING"
        print(f"  [{status}] {name}")
        if not ok:
            all_ok = False

    # 이미지 수 확인
    img_dir = os.path.join(scene_path, "images")
    if os.path.isdir(img_dir):
        n_images = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
        print(f"  Images: {n_images}")

    # GT mask 수 확인
    mask_dir = os.path.join(scene_path, "gt_mask")
    if os.path.isdir(mask_dir):
        n_masks = len([f for f in os.listdir(mask_dir) if f.endswith('.png')])
        print(f"  GT masks: {n_masks}")

    # 샘플 JSON 확인
    json_dir = os.path.join(scene_path, "json", "train_json")
    if not os.path.isdir(json_dir):
        json_dir = os.path.join(scene_path, "train_json")
    if os.path.isdir(json_dir):
        json_files = sorted([f for f in os.listdir(json_dir) if f.endswith('.json')])
        print(f"  Train JSONs: {len(json_files)}")
        if json_files:
            with open(os.path.join(json_dir, json_files[0])) as f:
                sample = json.load(f)
            print(f"  Sample JSON keys: {list(sample.keys())}")
            if 'object' in sample:
                obj = sample['object'][0]
                print(f"  Sample object keys: {list(obj.keys())}")
                if 'sentence' in obj:
                    print(f"  Sample sentences: {obj['sentence'][:2]}")

    return all_ok

# 선택한 장면 검증
if os.path.exists(SCENE_DATA_PATH):
    verify_scene(SCENE_DATA_PATH)
else:
    print(f"데이터 경로가 존재하지 않습니다: {SCENE_DATA_PATH}")

## 5. ReferSplat 학습 (Train)

**학습 흐름:**
1. 사전학습된 3DGS RGB 체크포인트 로드 (RGB 파라미터 freeze)
2. 각 Gaussian에 16차원 referring feature 추가
3. BERT로 텍스트 임베딩 → Position-aware Cross-Modal Interaction
4. BCE Loss + Contrastive Loss로 학습
5. 5 epoch, ~58분 (A6000 기준)

**주요 하이퍼파라미터:**
- referring feature dim: 16
- language_feature_lr: 0.0025
- mlp_lr / cross_attention_lr: 0.0001
- contrastive loss weight: 0.1
- top-τ schedule: 0.1 × 0.6^(iter/1000)

In [ ]:
%cd {REPO_DIR}
import glob

# 이미 학습된 체크포인트가 있으면 학습 건너뜀
existing_ckpts = sorted(glob.glob(os.path.join(SCENE_OUTPUT_PATH, "chkpnt*.pth")))
if existing_ckpts:
    print(f"학습된 체크포인트 발견! 학습을 건너뜁니다: {os.path.basename(existing_ckpts[-1])}")
    print("바로 섹션 6 (렌더링)으로 이동하세요.")
else:
    print(f"체크포인트 없음. 학습을 시작합니다.")
    !python train.py \
        -s {SCENE_DATA_PATH} \
        -m {SCENE_OUTPUT_PATH} \
        --start_checkpoint {PRETRAINED_CKPT}

### 5-1. (선택) 학습 모니터링

In [ ]:
# 저장된 체크포인트 확인
import glob

ckpt_files = sorted(glob.glob(os.path.join(SCENE_OUTPUT_PATH, "chkpnt*.pth")))
print(f"저장된 체크포인트 ({len(ckpt_files)}개):")
for f in ckpt_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")

## 6. 렌더링 (Render)

학습된 모델로 테스트 뷰에서 세그멘테이션 마스크를 렌더링합니다.

**테스트 프레임 인덱스:**
- ramen: [6, 24, 60, 65, 81, 119, 128]
- figurines: [83, 97, 146, 179]
- teatime: [2, 25, 43, 107, 129, 140]
- waldo: [19, 35, 67, 105, 162]

In [ ]:
%cd {REPO_DIR}

# render.py에서 model_path와 checkpoint 이름을 하드코딩하고 있으므로
# 직접 렌더링 함수를 호출합니다.

# 가장 최근 체크포인트 찾기
import glob
ckpt_files = sorted(glob.glob(os.path.join(SCENE_OUTPUT_PATH, "chkpnt*.pth")))
if ckpt_files:
    latest_ckpt = os.path.basename(ckpt_files[-1])
    print(f"Using checkpoint: {latest_ckpt}")
else:
    print("No checkpoint found! Train first.")
    latest_ckpt = None

In [ ]:
%%writefile {REPO_DIR}/render_colab.py
"""
Colab용 렌더링 스크립트 - 경로를 인자로 받음
"""
import re
import numpy as np
import torch
from scene import Scene
import os
from tqdm import tqdm
from os import makedirs
import torch.nn.functional as F
from gaussian_renderer import render
import torchvision
import random
from utils.general_utils import safe_state
from argparse import ArgumentParser
from arguments import ModelParams, PipelineParams, OptimizationParams
from gaussian_renderer import GaussianModel


def render_set(model_path, source_path, name, views, gaussians, pipeline, background, args):
    render_path = os.path.join(model_path, name, "renders")
    gts_path = os.path.join(model_path, name, "gt")
    render_npy_path = os.path.join(model_path, name, "renders_npy")
    gts_npy_path = os.path.join(model_path, name, "gt_npy")

    makedirs(render_npy_path, exist_ok=True)
    makedirs(gts_npy_path, exist_ok=True)
    makedirs(render_path, exist_ok=True)
    makedirs(gts_path, exist_ok=True)

    for idx, view in enumerate(tqdm(views, desc="Rendering progress")):
        for i in range(len(view.sentence)):
            sn = view.image_name
            number = re.findall(r'\d+', sn)
            number_int = int(number[0])
            output = render(view, gaussians, pipeline, background, args, sentence=view.sentence[i])
            rendering = output["language_feature_image"]
            rendering = torch.sigmoid(rendering)
            rendering = (rendering >= 0.5).float()
            gt = view.gt_mask[view.category[i]]
            fname = '{0:05d}'.format(number_int) + '{}'.format(view.category[i])
            np.save(os.path.join(render_npy_path, fname + ".npy"), rendering.permute(1, 2, 0).cpu().numpy())
            np.save(os.path.join(gts_npy_path, fname + ".npy"), gt.permute(1, 2, 0).cpu().numpy())
            torchvision.utils.save_image(rendering, os.path.join(render_path, fname + ".png"))
            torchvision.utils.save_image(gt, os.path.join(gts_path, fname + ".png"))


if __name__ == "__main__":
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    torch.cuda.manual_seed_all(0)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    parser = ArgumentParser()
    lp = ModelParams(parser)
    op = OptimizationParams(parser)
    pp = PipelineParams(parser)
    parser.add_argument("--checkpoint_file", type=str, required=True)
    parser.add_argument("--quiet", action="store_true")
    args = parser.parse_args()
    args.include_feature = True

    safe_state(args.quiet)

    with torch.no_grad():
        dataset = lp.extract(args)
        gaussians = GaussianModel(dataset.sh_degree)
        scene = Scene(dataset, gaussians, shuffle=False)

        (model_params, first_iter) = torch.load(args.checkpoint_file, map_location=f'cuda:{torch.cuda.current_device()}')
        gaussians.restore(model_params, args, mode='test')

        bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
        background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

        render_set(dataset.model_path, dataset.source_path, "test_results",
                   scene.getTestCameras(), gaussians, pp.extract(args), background, args)

    print("Rendering complete!")

In [ ]:
%cd {REPO_DIR}

if latest_ckpt:
    CKPT_PATH = os.path.join(SCENE_OUTPUT_PATH, latest_ckpt)
    !python render_colab.py \
        -s {SCENE_DATA_PATH} \
        -m {SCENE_OUTPUT_PATH} \
        --checkpoint_file {CKPT_PATH}
else:
    print("체크포인트가 없습니다. 먼저 학습을 실행하세요.")

## 7. 평가 (mIoU)

In [ ]:
import torch
import numpy as np
from PIL import Image

def calculate_iou(pred_mask, gt_mask):
    intersection = torch.logical_and(pred_mask, gt_mask).sum().float()
    union = torch.logical_or(pred_mask, gt_mask).sum().float()
    if union == 0:
        return 0.0
    return (intersection / union).item()

def load_mask(file_path):
    mask = Image.open(file_path).convert("L")
    mask = np.array(mask)
    mask = torch.from_numpy(mask).float()
    return mask > 0

def evaluate_miou(render_dir, gt_dir):
    """렌더링 결과와 GT 간 mIoU 계산"""
    iou_list = []
    details = []

    for filename in sorted(os.listdir(render_dir)):
        if not filename.endswith(".png"):
            continue
        render_path = os.path.join(render_dir, filename)
        gt_path = os.path.join(gt_dir, filename)
        if not os.path.exists(gt_path):
            print(f"  [SKIP] GT not found: {filename}")
            continue

        pred_mask = load_mask(render_path)
        gt_mask = load_mask(gt_path)

        if gt_mask.sum() == 0:
            continue

        iou = calculate_iou(pred_mask, gt_mask)
        iou_list.append(iou)
        details.append((filename, iou))

    miou = np.mean(iou_list) if iou_list else 0.0
    return miou, details

# 평가 실행
render_dir = os.path.join(SCENE_OUTPUT_PATH, "test_results", "renders")
gt_dir = os.path.join(SCENE_OUTPUT_PATH, "test_results", "gt")

if os.path.exists(render_dir) and os.path.exists(gt_dir):
    miou, details = evaluate_miou(render_dir, gt_dir)
    print(f"\n{'=' * 50}")
    print(f"Scene: {SCENE}")
    print(f"mIoU: {miou * 100:.2f}%")
    print(f"{'=' * 50}")
    print(f"\nPer-query IoU:")
    for fname, iou in details:
        print(f"  {fname}: {iou * 100:.1f}%")
else:
    print(f"렌더링 결과가 없습니다. 먼저 렌더링을 실행하세요.")
    print(f"  Expected: {render_dir}")

## 8. 시각화

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

def visualize_results(render_dir, gt_dir, n_samples=4):
    """렌더링 결과 시각화 (Predicted vs GT)"""
    render_files = sorted([f for f in os.listdir(render_dir) if f.endswith('.png')])
    n_show = min(n_samples, len(render_files))

    if n_show == 0:
        print("시각화할 결과가 없습니다.")
        return

    fig, axes = plt.subplots(n_show, 2, figsize=(10, 4 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for i, fname in enumerate(render_files[:n_show]):
        pred = np.array(Image.open(os.path.join(render_dir, fname)).convert("L"))
        gt_path = os.path.join(gt_dir, fname)
        gt = np.array(Image.open(gt_path).convert("L")) if os.path.exists(gt_path) else np.zeros_like(pred)

        axes[i, 0].imshow(pred, cmap='gray')
        axes[i, 0].set_title(f"Predicted: {fname}")
        axes[i, 0].axis('off')

        axes[i, 1].imshow(gt, cmap='gray')
        axes[i, 1].set_title(f"Ground Truth: {fname}")
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

render_dir = os.path.join(SCENE_OUTPUT_PATH, "test_results", "renders")
gt_dir = os.path.join(SCENE_OUTPUT_PATH, "test_results", "gt")

if os.path.exists(render_dir):
    visualize_results(render_dir, gt_dir)
else:
    print("렌더링 결과가 없습니다.")

## 9. (선택) Two-Stage 최적화

Stage 1에서 렌더링된 마스크를 새로운 pseudo GT로 사용하여 재학습합니다.
논문 기준 ramen: 35.2 → 36.9 mIoU 향상.

In [ ]:
# Two-stage 최적화를 위해:
# 1. Stage 1의 렌더링 마스크를 gt_mask 폴더로 복사
# 2. 동일 파이프라인으로 재학습

# STAGE2_OUTPUT = os.path.join(OUTPUT_ROOT, f"{SCENE}_stage2")
# STAGE1_CKPT = os.path.join(SCENE_OUTPUT_PATH, latest_ckpt)  # Stage1 결과

# !python train.py \
#     -s {SCENE_DATA_PATH} \
#     -m {STAGE2_OUTPUT} \
#     --start_checkpoint {PRETRAINED_CKPT}

print("Two-stage 최적화는 선택사항입니다. 주석을 해제하고 실행하세요.")

## 10. 전체 장면 일괄 실행

4개 장면 모두 학습 + 평가하려면 아래를 실행하세요.

In [ ]:
# 전체 장면 일괄 학습 및 평가
# 주의: 4개 장면 × ~58분 = 약 4시간 소요

ALL_SCENES = ["ramen", "figurines", "teatime", "waldo_kitchen"]
# data_dict의 키와 폴더명 매핑 (waldo_kitchen -> waldo)
SCENE_KEY_MAP = {
    "ramen": "ramen",
    "figurines": "figurines", 
    "teatime": "teatime",
    "waldo_kitchen": "waldo",
}

results = {}

# for scene in ALL_SCENES:
#     print(f"\n{'='*60}")
#     print(f"Processing scene: {scene}")
#     print(f"{'='*60}")
#     
#     scene_data = os.path.join(DATA_ROOT, scene)
#     scene_output = os.path.join(OUTPUT_ROOT, scene)
#     scene_ckpt = os.path.join(CHECKPOINT_ROOT, scene, "chkpnt30000.pth")
#     
#     # 학습
#     !cd {REPO_DIR} && python train.py -s {scene_data} -m {scene_output} --start_checkpoint {scene_ckpt}
#     
#     # 렌더링
#     ckpt_files = sorted(glob.glob(os.path.join(scene_output, "chkpnt*.pth")))
#     if ckpt_files:
#         !cd {REPO_DIR} && python render_colab.py -s {scene_data} -m {scene_output} --checkpoint_file {ckpt_files[-1]}
#     
#     # 평가
#     r_dir = os.path.join(scene_output, "test_results", "renders")
#     g_dir = os.path.join(scene_output, "test_results", "gt")
#     if os.path.exists(r_dir):
#         miou, _ = evaluate_miou(r_dir, g_dir)
#         results[scene] = miou
#         print(f"  {scene} mIoU: {miou*100:.2f}%")

# if results:
#     avg = np.mean(list(results.values()))
#     print(f"\n{'='*60}")
#     print("Final Results:")
#     for s, v in results.items():
#         print(f"  {s}: {v*100:.2f}%")
#     print(f"  Average: {avg*100:.2f}%")
#     print(f"{'='*60}")

print("전체 장면 일괄 실행은 주석을 해제하고 실행하세요.")
print("예상 소요 시간: 약 4시간 (A100 기준)")

---
## 참고: 논문 기대 성능 (mIoU)

### R3DGS (Ref-LERF)
| Scene | ramen | figurines | teatime | waldo_kitchen | avg |
|---|---|---|---|---|---|
| ReferSplat | 35.2 | 25.7 | 31.3 | 24.4 | 29.2 |

### Open-Vocabulary (LERF-OVS)
| Scene | ramen | figurines | teatime | waldo_kitchen | avg |
|---|---|---|---|---|---|
| ReferSplat | 55.1 | 67.5 | 50.1 | 48.9 | 55.4 |